# 📖 ASL Word Training Pipeline (WLASL → BiLSTM)

**Purpose:** Download WLASL videos, extract MediaPipe hand landmarks,
build 30-frame sequences, and train a BiLSTM model for ASL word recognition.

**Output:** `asl_word_sequences.npz` + `asl_word_lstm_model_best.h5`

---

## 1. Setup & Configuration

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES & CONFIGURE
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['kagglehub', 'mediapipe==0.10.33', 'scikit-learn', 'tqdm']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

import os, json, cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import kagglehub
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# --- GPU Configuration ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f'GPU mode: {len(gpus)} GPU(s), mixed_float16')
else:
    print('CPU mode')

# --- Paths ---
PROJECT_DIR = Path('.').resolve()
OUTPUT_NPZ = PROJECT_DIR / 'asl_word_sequences.npz'
MODEL_OUTPUT = PROJECT_DIR / 'asl_word_lstm_model_best.h5'

# --- Hyperparameters ---
SEQUENCE_LENGTH = 30     # frames per sequence
NUM_FEATURES = 78        # 21 landmarks * 3 coords
MAX_WORDS = 157          # target word count
MAX_VIDEOS_PER_WORD = 50 # cap per class
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
print(f'Project: {PROJECT_DIR}')
print('Configuration loaded')


## 2. Download WLASL Dataset from Kaggle

Uses `kagglehub` to download the pre-processed WLASL dataset which includes
both the metadata JSON and pre-downloaded videos (~5.4 GB total).

In [ ]:
# ============================================================
# CELL 2: DOWNLOAD WLASL DATASET FROM KAGGLE
# ============================================================
# Dataset: https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed
# Downloads ~5.4 GB to kagglehub cache (only once, reuses cache after)

print('Downloading WLASL-processed dataset from Kaggle...')
print('(First run downloads ~5.4 GB. Re-runs use cache instantly.)')

dataset_path = Path(kagglehub.dataset_download('risangbaskoro/wlasl-processed'))
print(f'Dataset path: {dataset_path}')

# --- Locate files ---
# Try nslt_2000.json first, then WLASL_v0.3.json
WLASL_JSON = None
for name in ['nslt_2000.json', 'WLASL_v0.3.json']:
    candidate = dataset_path / name
    if candidate.exists():
        WLASL_JSON = candidate
        break
    # Search recursively
    for found in dataset_path.rglob(name):
        WLASL_JSON = found
        break
    if WLASL_JSON:
        break

# Find videos directory
VIDEO_DIR = dataset_path / 'videos'
if not VIDEO_DIR.exists():
    for found in dataset_path.rglob('videos'):
        if found.is_dir():
            VIDEO_DIR = found
            break

print(f'JSON: {WLASL_JSON} (exists: {WLASL_JSON is not None and WLASL_JSON.exists()})')
print(f'Videos: {VIDEO_DIR} (exists: {VIDEO_DIR.exists()})')

# Count actual video files
video_files = list(VIDEO_DIR.glob('*.mp4')) if VIDEO_DIR.exists() else []
print(f'Video files found: {len(video_files)}')
if video_files:
    print(f'Sample filenames: {[v.name for v in video_files[:5]]}')

# --- Load and parse JSON ---
with open(WLASL_JSON, 'r') as f:
    wlasl_data = json.load(f)

print(f'JSON entries: {len(wlasl_data)}')

# Detect JSON format and build word_info
# Format A (WLASL_v0.3.json): [{gloss: 'book', instances: [{video_id: '00295', ...}]}, ...]
# Format B (nslt_2000.json):   Same structure as v0.3
word_info = []

if isinstance(wlasl_data, list) and len(wlasl_data) > 0:
    sample = wlasl_data[0]
    print(f'JSON format: keys = {list(sample.keys())}')
    
    if 'gloss' in sample:
        # Standard WLASL format
        for entry in wlasl_data:
            gloss = entry['gloss']
            instances = entry.get('instances', [])
            # Check which videos actually exist
            available = 0
            for inst in instances:
                vid_id = str(inst.get('video_id', ''))
                # Try multiple filename patterns
                found = False
                for pattern in [f'{vid_id}.mp4', f'{vid_id.zfill(5)}.mp4']:
                    if (VIDEO_DIR / pattern).exists():
                        found = True
                        break
                if found:
                    available += 1
            word_info.append((gloss, available, instances))
    else:
        print(f'Unknown JSON format. First entry: {sample}')
        raise ValueError('Cannot parse JSON — unknown format')

# Sort by available videos, take top MAX_WORDS
word_info.sort(key=lambda x: x[1], reverse=True)
selected_words = [w for w in word_info if w[1] > 0][:MAX_WORDS]

# Build mapping
word_to_id = {}
id_to_word = {}
for i, (gloss, count, _) in enumerate(selected_words):
    word_to_id[gloss] = i
    id_to_word[i] = gloss

print(f'\nSelected {len(selected_words)} words with available videos:')
for i, (gloss, count, _) in enumerate(selected_words[:10]):
    print(f'  {i}: {gloss} ({count} videos)')
if len(selected_words) > 10:
    print(f'  ... and {len(selected_words)-10} more')

total_vids = sum(min(c, MAX_VIDEOS_PER_WORD) for _, c, _ in selected_words)
print(f'\nTotal videos to process: ~{total_vids}')


## 3. Map Local Videos

Videos are already included in the Kaggle dataset — no download needed.
This cell maps each word to its available video files.

In [ ]:
# ============================================================
# CELL 3: BUILD VIDEO MANIFEST FROM KAGGLE DATASET
# ============================================================
# Map each word to its locally available video files.

download_manifest = {}  # word_id -> [video_paths]
total_found = 0
total_missing = 0

for gloss, avail_count, instances in tqdm(selected_words, desc='Mapping videos'):
    word_id = word_to_id[gloss]
    paths = []
    
    for inst in instances[:MAX_VIDEOS_PER_WORD]:
        vid_id = str(inst.get('video_id', ''))
        
        # Try multiple filename patterns
        vid_path = None
        for pattern in [f'{vid_id}.mp4', f'{vid_id.zfill(5)}.mp4']:
            candidate = VIDEO_DIR / pattern
            if candidate.exists():
                vid_path = candidate
                break
        
        if vid_path:
            paths.append(vid_path)
            total_found += 1
        else:
            total_missing += 1
    
    download_manifest[word_id] = paths

print(f'\nVideos found: {total_found}')
print(f'Videos missing: {total_missing}')
print(f'Words with 1+ video: {sum(1 for p in download_manifest.values() if p)}')


## 4. MediaPipe Landmark Extraction

In [ ]:
# ============================================================
# CELL 4: EXTRACT MEDIAPIPE HAND LANDMARKS FROM VIDEOS
# ============================================================
import numpy as np
import cv2

mp_hands = mp.solutions.hands

def compute_angle(a, b, c):
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.arccos(cosine)

ANGLE_TRIPLETS = [
    (0, 1, 2), (1, 2, 3), (2, 3, 4),
    (0, 5, 6), (5, 6, 7), (6, 7, 8),
    (0, 9, 10), (9, 10, 11), (10, 11, 12),
    (0, 13, 14), (13, 14, 15), (14, 15, 16),
    (0, 17, 18), (17, 18, 19), (18, 19, 20),
]

def extract_landmarks_from_video(video_path, hands_detector):
    """
    Extract per-frame hand landmarks from a video.
    Returns list of 78-dim arrays (one per frame where hand was detected).
    Uses forward-fill for frames with no detection.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    
    all_landmarks = []
    last_valid = None
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands_detector.process(rgb)
        
        if results.multi_hand_landmarks:
            hand = results.multi_hand_landmarks[0]
            raw_pts = np.array([[p.x, p.y, p.z] for p in hand.landmark], dtype=np.float32)
            
            # 1. Wrist-relative normalization
            wrist = raw_pts[0].copy()
            relative = raw_pts - wrist
            rel_flat = relative.flatten()
            
            # 2. Joint angles
            angles = []
            for a_idx, b_idx, c_idx in ANGLE_TRIPLETS:
                angle = compute_angle(raw_pts[a_idx], raw_pts[b_idx], raw_pts[c_idx])
                angles.append(angle)
            angles = np.array(angles, dtype=np.float32)
            
            # 3. Combine
            features = np.concatenate([rel_flat, angles])
            last_valid = features
            all_landmarks.append(features)
        elif last_valid is not None:
            all_landmarks.append(last_valid.copy())  # forward-fill
        # else: skip frame (no detection yet)
    
    cap.release()
    return all_landmarks if len(all_landmarks) >= 5 else None

print('✅ Extraction function defined (78-dim)')

## 5. Build 30-Frame Sequences & Save NPZ

In [ ]:
# ============================================================
# CELL 5: RESAMPLE TO FIXED LENGTH & BUILD DATASET
# ============================================================

def resample_sequence(landmarks_list, target_length=30):
    """Resample variable-length sequence to fixed length using uniform sampling."""
    n = len(landmarks_list)
    if n == target_length:
        return np.array(landmarks_list)
    indices = np.linspace(0, n - 1, target_length).astype(int)
    return np.array([landmarks_list[i] for i in indices])

# Process all videos
all_X = []
all_y = []
skipped = 0

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    model_complexity=0,
    min_detection_confidence=0.5
) as hands:
    for word_id, video_paths in tqdm(download_manifest.items(), desc='Extracting landmarks'):
        for vpath in video_paths:
            landmarks = extract_landmarks_from_video(vpath, hands)
            if landmarks is None:
                skipped += 1
                continue
            
            seq = resample_sequence(landmarks, SEQUENCE_LENGTH)
            assert seq.shape == (SEQUENCE_LENGTH, NUM_FEATURES), f'Bad shape: {seq.shape}'
            all_X.append(seq)
            all_y.append(word_id)

X = np.array(all_X, dtype=np.float32)
y = np.array(all_y, dtype=np.int32)

print(f'\n✅ Dataset built!')
print(f'   X shape: {X.shape}  (sequences, frames, features)')
print(f'   y shape: {y.shape}')
print(f'   Classes: {len(np.unique(y))}')
print(f'   Skipped: {skipped} videos (too few frames / no hand)')

# Save NPZ
np.savez_compressed(OUTPUT_NPZ, X=X, y=y)
print(f'\n💾 Saved: {OUTPUT_NPZ}')
print(f'   File size: {OUTPUT_NPZ.stat().st_size / 1024 / 1024:.1f} MB')


## 6. Verify Dataset

In [ ]:
# ============================================================
# CELL 6: QUALITY CHECK
# ============================================================
data = np.load(OUTPUT_NPZ)
X_check, y_check = data['X'], data['y']

print(f'📋 DATASET VERIFICATION')
print(f'   Shape: X={X_check.shape}, y={y_check.shape}')
print(f'   NaN count: {np.isnan(X_check).sum()}')
print(f'   Feature range: [{X_check.min():.4f}, {X_check.max():.4f}]')
print(f'   Classes: {len(np.unique(y_check))}')
print(f'   Samples per class (min/max/mean): '
      f'{np.bincount(y_check).min()}/{np.bincount(y_check).max()}/'
      f'{np.bincount(y_check).mean():.1f}')
print(f'✅ Verification complete')


## 7. BiLSTM Model Training

Architecture: ~320K params BiLSTM, matching the integration spec.

In [ ]:
# ============================================================
# CELL 7: TRAIN BiLSTM MODEL (Enhanced Regularization)
# ============================================================
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Bidirectional, LSTM, Dense, Dropout, BatchNormalization, Input, SpatialDropout1D
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.utils import class_weight

# Load data
data = np.load(OUTPUT_NPZ)
X_all, y_all = data['X'], data['y']
num_classes = len(np.unique(y_all))

# Train/val/test split (64/16/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)

# One-hot encode
y_train_oh = to_categorical(y_train, num_classes)
y_val_oh = to_categorical(y_val, num_classes)
y_test_oh = to_categorical(y_test, num_classes)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Classes: {num_classes}')

# Class weights for imbalanced data
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_array = np.clip(class_weights_array, 0.5, 10.0)
class_weights = dict(enumerate(class_weights_array))

# Build BiLSTM with advanced regularization
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES)),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(1e-4))),
    BatchNormalization(),
    Dropout(0.3),
    Bidirectional(LSTM(64, kernel_regularizer=tf.keras.regularizers.l2(1e-4))),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    Dropout(0.2),
    Dense(num_classes, activation='softmax', dtype='float32'),
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)
model.summary()

callbacks = [
    ModelCheckpoint(str(MODEL_OUTPUT), monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-7, verbose=1),
]

history = model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=120,
    batch_size=64,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print('\n✅ Training complete')

## 8. Evaluation

In [ ]:
# ============================================================
# CELL 8: EVALUATE MODEL
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Load best model
best_model = tf.keras.models.load_model(str(MODEL_OUTPUT))

# Test accuracy
test_loss, test_acc = best_model.evaluate(X_test, y_test_oh, verbose=0)
print(f'\n📊 Test Accuracy: {test_acc:.4f}')
print(f'📊 Test Loss: {test_loss:.4f}')

# Classification report
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
print('\n📋 Classification Report (top classes):')
report = classification_report(y_test, y_pred, output_dict=True)
# Show top 20 by f1
class_metrics = [(k, v['f1-score'], v['support'])
                 for k, v in report.items() if k.isdigit()]
class_metrics.sort(key=lambda x: x[1], reverse=True)
for cls_id, f1, sup in class_metrics[:20]:
    word = id_to_word.get(int(cls_id), cls_id)
    print(f'   {word:20s} F1={f1:.3f} (n={int(sup)})')

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

print('\n✅ Evaluation complete')


## 9. Quick Inference Test

In [ ]:
# ============================================================
# CELL 9: QUICK INFERENCE TEST
# ============================================================
idx = np.random.randint(len(X_test))
sample = X_test[idx:idx+1]
true_label = y_test[idx]

pred = best_model.predict(sample, verbose=0)
pred_label = np.argmax(pred)
pred_conf = pred[0][pred_label]

print(f'🎯 True word:      {id_to_word.get(true_label, true_label)}')
print(f'🤖 Predicted word: {id_to_word.get(pred_label, pred_label)}')
print(f'📊 Confidence:     {pred_conf:.4f}')
print(f'✅ {"CORRECT" if true_label == pred_label else "INCORRECT"}')

# Save word mapping
import json
mapping_path = PROJECT_DIR / 'asl_word_labels.json'
with open(mapping_path, 'w') as f:
    json.dump(id_to_word, f, indent=2)
print(f'\n💾 Word labels saved: {mapping_path}')
print('\n✅ Pipeline complete! Files produced:')
print(f'   📦 {OUTPUT_NPZ}')
print(f'   🧠 {MODEL_OUTPUT}')
print(f'   🏷️  {mapping_path}')
